# Detect spots: .ome.tiff -> H5 -> ilastik

In [12]:
import subprocess
from pathlib import Path

import h5py
import numpy as np
import vigra
from aicsimageio import AICSImage

input_dir = Path("/srv/scratch/berrylab/z3536241/NikonSpinningDisk/260515_mACPOLR2A_CRC_EU/20260519_171947_338/OME-TIFF-MIP/")
tmp_dir = Path("/srv/scratch/berrylab/z3532965/tmp")
tmp_dir.mkdir(parents=True, exist_ok=True)

ilastik_path = "/srv/scratch/z3532965/ilastik/ilastik-1.4.2rc1-Linux/run_ilastik.sh"
ilastik_project = "/srv/scratch/z3532965/src/publications/2026_POLR2A_homeostasis/Nucleolus/nucleolus_from_DAPI_POLR2A_3class.ilp"

channels = [1, 2]

In [10]:
files = sorted(input_dir.glob("*.ome.tiff"))
print(f"Found {len(files)} .ome.tiff files")
assert files, "No .ome.tiff files found"

Found 960 .ome.tiff files


In [11]:
axistags = vigra.defaultAxistags("tzyxc")
h5_paths = []

for f in files:
    img = AICSImage(f)
    data = img.get_image_data("CYX", T=0, Z=0)  # CYX
    data = data[channels, :, :]  # keep channels 1 and 2

    h5_path = tmp_dir / (f.name.replace(".ome.tiff", "") + ".h5")
    image_yxc = np.transpose(data, (1, 2, 0))  # CYX -> YXC
    image_5d = image_yxc[np.newaxis, np.newaxis, :, :, :]  # TZYXC

    with h5py.File(h5_path, "w") as h5_file:
        ds = h5_file.create_dataset(
            name="data", data=image_5d, chunks=(1, 1, 64, 64, 1)
        )
        ds.attrs["axistags"] = axistags.toJSON()
    h5_paths.append(str(h5_path))

print(f"Wrote {len(h5_paths)} H5 files to {tmp_dir}")

Wrote 960 H5 files to /srv/scratch/berrylab/z3532965/tmp


In [13]:
h5_paths

['/srv/scratch/berrylab/z3532965/tmp/WellC03_Channel647,405,561,XXX_Seq0000_0001.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC03_Channel647,405,561,XXX_Seq0000_0002.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC03_Channel647,405,561,XXX_Seq0000_0003.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC03_Channel647,405,561,XXX_Seq0000_0004.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC04_Channel647,405,561,XXX_Seq0001_0001.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC04_Channel647,405,561,XXX_Seq0001_0002.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC04_Channel647,405,561,XXX_Seq0001_0003.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC04_Channel647,405,561,XXX_Seq0001_0004.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC05_Channel647,405,561,XXX_Seq0002_0001.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC05_Channel647,405,561,XXX_Seq0002_0002.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC05_Channel647,405,561,XXX_Seq0002_0003.h5',
 '/srv/scratch/berrylab/z3532965/tmp/WellC05_Channel64

In [26]:
cmd = [ilastik_path, "--headless", "--project", ilastik_project] + [str(tmp_dir) + "/*.h5"]

In [27]:
print(" ".join(cmd))

/srv/scratch/z3532965/ilastik/ilastik-1.4.2rc1-Linux/run_ilastik.sh --headless --project /srv/scratch/z3532965/src/publications/2026_POLR2A_homeostasis/Nucleolus/nucleolus_from_DAPI_POLR2A_3class.ilp /srv/scratch/berrylab/z3532965/tmp/*.h5


In [ ]:
subprocess.run(["rm"] + h5_paths)
print("Removed temporary H5 files")